In [1]:
import os, glob
import numpy as np
import pandas as pd
import xarray as xr

NETID = "k16v981"
BASE = f"/home/{NETID}/my_work/data/era5/era5_sst/"
FILES = sorted(glob.glob(os.path.join(BASE, "era5_sst_*.nc")))

OUT = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/"
OUT_CSV = os.path.join(OUT, "roni_dmi_monthly_1950_2025.csv")

CLIM_START = "1991-01-01"
CLIM_END   = "2020-12-31"
START = "1950-01-01"
END   = "2025-12-31"

# Boxes (lat/lon in degrees). We'll handle lon wrap.
NINO34 = dict(lat_min=-5.0,  lat_max=5.0,  lon_min=-170.0, lon_max=-120.0)
TROP20 = dict(lat_min=-20.0, lat_max=20.0, lon_min=-180.0, lon_max=180.0)
WTIO   = dict(lat_min=-10.0, lat_max=10.0, lon_min=50.0,  lon_max=70.0)
SETIO  = dict(lat_min=-10.0, lat_max=0.0,  lon_min=90.0,  lon_max=110.0)

def _pick_sst_var(ds):
    for v in ds.data_vars:
        nm = v.lower()
        if nm in ("sst", "sea_surface_temperature") or "sst" in nm or "sea_surface_temperature" in nm:
            return ds[v]
    return ds[list(ds.data_vars)[0]]

def _fix_time_and_expver(ds):
    if "valid_time" in ds.coords or "valid_time" in ds.dims:
        ds = ds.rename({"valid_time": "time"})

    # If expver exists as a dim, pick expver=1 if present else first
    if "expver" in ds.dims:
        if "expver" in ds.coords and np.any(ds["expver"].values == 1):
            ds = ds.sel(expver=1)
        else:
            ds = ds.isel(expver=0)
        ds = ds.drop_vars("expver", errors="ignore")

    # drop if it exists as a coord/var but not a dim
    if "expver" in ds.coords and "expver" not in ds.dims:
        ds = ds.drop_vars("expver", errors="ignore")
    return ds

def _standardize_lon(da):
    lon_name = "longitude" if "longitude" in da.coords else "lon"
    lon = da[lon_name]
    if lon.max() > 180:
        lon_new = ((lon + 180) % 360) - 180
        da = da.assign_coords({lon_name: lon_new}).sortby(lon_name)
    return da

def _subset_box(da, box):
    lat_name = "latitude" if "latitude" in da.coords else "lat"
    lon_name = "longitude" if "longitude" in da.coords else "lon"

    lat = da[lat_name]
    if lat[0] > lat[-1]:
        da = da.sel({lat_name: slice(box["lat_max"], box["lat_min"])})
    else:
        da = da.sel({lat_name: slice(box["lat_min"], box["lat_max"])})

    da = da.sel({lon_name: slice(box["lon_min"], box["lon_max"])})
    return da

def _area_weighted_mean(da):
    lat_name = "latitude" if "latitude" in da.coords else "lat"
    lon_name = "longitude" if "longitude" in da.coords else "lon"
    w = np.cos(np.deg2rad(da[lat_name]))
    return da.weighted(w).mean(dim=(lat_name, lon_name))

# ---- build raw monthly means (Kelvin) for the four regions ----
records = []

for f in FILES:
    ds = xr.open_dataset(f)  # no dask; small slices are fast
    ds = _fix_time_and_expver(ds)
    sst = _pick_sst_var(ds)
    sst = _standardize_lon(sst)

    # clip time within file (cheap)
    sst = sst.sel(time=slice(START, END))

    # compute 1D monthly means for each region (12 vals per year file)
    nino = _area_weighted_mean(_subset_box(sst, NINO34)).to_series()
    trop = _area_weighted_mean(_subset_box(sst, TROP20)).to_series()
    wtio = _area_weighted_mean(_subset_box(sst, WTIO)).to_series()
    seti = _area_weighted_mean(_subset_box(sst, SETIO)).to_series()

    df = pd.concat([nino, trop, wtio, seti], axis=1)
    df.columns = ["nino34", "trop20", "wtio", "setio"]
    records.append(df)

    ds.close()

raw = pd.concat(records).sort_index()
raw.index = pd.to_datetime(raw.index).to_period("M").to_timestamp()
raw = raw.loc["1950-01-01":"2025-12-01"].copy()

# ---- anomalies computed on 1D series (super fast) ----
baseline = raw.loc[CLIM_START:CLIM_END].copy()
clim = baseline.groupby(baseline.index.month).mean()  # 12x4

anom = raw.copy()
for m in range(1, 13):
    anom.loc[anom.index.month == m, :] = raw.loc[raw.index.month == m, :] - clim.loc[m, :].values

# ---- indices (Kelvin anomalies == Celsius anomalies) ----
roni = anom["nino34"] - anom["trop20"]
dmi  = anom["wtio"]   - anom["setio"]

out = pd.DataFrame({"time": roni.index, "RONI": roni.values, "DMI": dmi.values})
out.to_csv(OUT_CSV, index=False)
print("✅ Wrote:", OUT_CSV)
print(out.head())
print(out.tail())


ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed


✅ Wrote: /home/k16v981/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/roni_dmi_monthly_1950_2025.csv
        time      RONI       DMI
0 1950-01-01 -0.504667  0.103981
1 1950-01-01 -0.504284  0.103981
2 1950-01-01 -0.505611  0.103981
3 1950-01-01 -0.505558  0.103981
4 1950-01-01 -0.496440  0.094247
             time      RONI       DMI
103787 2025-08-01 -0.544987 -0.918548
103788 2025-09-01 -0.630689 -1.005824
103789 2025-10-01 -0.720303 -1.500266
103790 2025-11-01 -0.791258 -0.842156
103791 2025-12-01 -0.769086 -0.217091
